# CTDenoiser — Training Notebook

| Part | What happens |
|------|-------------|
| **0 — Setup** | Mount Drive, clone/update repo, install deps |
| **1 — Smoke test** | Verify everything works on synthetic data |
| **2 — Train** | Train RED-CNN and CTformer on real data |
| **3 — W&B sweep** | Bayesian HP search over both architectures |

**Prerequisites:** Run `00_build_cache.ipynb` once to download the TCIA LDCT data and build `ldct_cache.h5` on your Drive.

---
## Part 0 — Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL  = 'https://github.com/tsereda/CTDenoiser.git'
BRANCH    = 'main'
REPO_ROOT = '/content/drive/MyDrive/CTDenoiser'

if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
    os.makedirs(REPO_ROOT, exist_ok=True)
    !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
    print('Cloned.')
else:
    print('Repo already exists, pulling latest.')

%cd {REPO_ROOT}
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}

In [ ]:
%cd {REPO_ROOT}
!pip install -q -r requirements.txt

In [ ]:
%cd {REPO_ROOT}
!pytest -q

---
## Part 1 — Smoke Test (Synthetic Data)

Verifies the training loop runs end-to-end without needing real CT data.

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model redcnn --epochs 1 --synthetic-len 32
print('RED-CNN smoke test passed.')

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train --model ctformer --epochs 1 --synthetic-len 32
print('CTformer smoke test passed.')

---
## Part 2 — Train RED-CNN and CTformer on Real Data

Copies the HDF5 cache to fast local disk, then trains both models sequentially.
Adjust `--epochs` and `--batch-size` to your GPU budget.

In [ ]:
import os, shutil

DRIVE_CACHE = '/content/drive/MyDrive/ldct_cache.h5'
LOCAL_CACHE = '/content/ldct_cache.h5'

if os.path.exists(DRIVE_CACHE) and not os.path.exists(LOCAL_CACHE):
    print('Copying cache to local disk for faster I/O...')
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    print('Done.')

assert os.path.exists(LOCAL_CACHE), (
    'ldct_cache.h5 not found — run 00_build_cache.ipynb first, '
    'or place the file at ' + DRIVE_CACHE
)

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model redcnn \
    --h5-cache /content/ldct_cache.h5 \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

In [ ]:
%cd {REPO_ROOT}
!python -m ctdenoiser.train \
    --model ctformer \
    --h5-cache /content/ldct_cache.h5 \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64 \
    --lr 1e-4

---
## Part 3 — Weights & Biases Hyperparameter Sweep

Bayesian search over learning rate, batch size, patch size, **and model architecture**
(RED-CNN vs CTformer). Each run is logged to your W&B project so you can compare
them side-by-side.

**Before running:** log in to W&B once with `wandb login` or set `WANDB_API_KEY`.

In [ ]:
import wandb
wandb.login()   # prompts for API key if not already set

In [ ]:
SWEEP_PROJECT = 'ctdenoiser'

# How many total runs to launch (increase for a thorough search)
SWEEP_COUNT = 20

# Uses the same local cache copied in Part 2 above
SWEEP_H5_CACHE = LOCAL_CACHE

sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val/psnr', 'goal': 'maximize'},
    'parameters': {
        'model':      {'values': ['redcnn', 'ctformer']},
        'lr':         {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-3},
        'batch_size': {'values': [8, 16, 32]},
        'patch_size': {'values': [64, 96]},
        'epochs':     {'value': 5},
    },
}

sweep_id = wandb.sweep(sweep_config, project=SWEEP_PROJECT)
print(f'Sweep created: {sweep_id}')

In [ ]:
import sys, os
sys.path.insert(0, REPO_ROOT)

import torch
from torch.utils.data import DataLoader
import wandb

from ctdenoiser.models import CTformer, REDCNN
from ctdenoiser.data.dataset import HDF5CTDataset, SyntheticCTDataset
from ctdenoiser.inference import overlapped_inference
from ctdenoiser.metrics import psnr, rmse, ssim

MODELS = {'ctformer': CTformer, 'redcnn': REDCNN}


def train_run():
    """Single W&B sweep run: build loaders, train, log metrics per epoch."""
    run = wandb.init()
    cfg = run.config

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = MODELS[cfg.model]().to(device)

    wandb.config.update({'device': str(device)}, allow_val_change=True)

    # ── Data ────────────────────────────────────────────────────────────────
    h5_path = SWEEP_H5_CACHE
    if h5_path and os.path.exists(h5_path):
        train_pids, val_pids = HDF5CTDataset.split_patients(h5_path, val_fraction=0.2, seed=0)
        train_ds = HDF5CTDataset(h5_path, train_pids, patch_size=cfg.patch_size, train=True)
        val_ds   = HDF5CTDataset(h5_path, val_pids,   patch_size=cfg.patch_size, train=False)
        full_slice_eval = True
    else:
        print('H5 cache not found — using synthetic data.')
        train_ds = SyntheticCTDataset(length=128, patch_size=cfg.patch_size)
        val_ds   = SyntheticCTDataset(length=32,  patch_size=cfg.patch_size)
        full_slice_eval = False

    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

    # ── Training + per-epoch validation ─────────────────────────────────────
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    criterion = torch.nn.MSELoss()
    n_train   = len(train_loader.dataset)

    for epoch in range(1, cfg.epochs + 1):
        # Train
        model.train()
        running = 0.0
        for low, full in train_loader:
            low, full = low.to(device), full.to(device)
            optimizer.zero_grad()
            loss = criterion(model(low), full)
            loss.backward()
            optimizer.step()
            running += loss.item() * low.size(0)
        train_loss = running / n_train

        # Validate
        model.eval()
        n, p, s, r = 0, 0.0, 0.0, 0.0
        with torch.no_grad():
            for low, full in val_loader:
                low, full = low.to(device), full.to(device)
                if full_slice_eval:
                    pred = overlapped_inference(
                        model, low,
                        patch_size=cfg.patch_size,
                        margin=cfg.patch_size // 4,
                    ).clamp(0.0, 1.0)
                else:
                    pred = model(low).clamp(0.0, 1.0)
                bs = low.size(0)
                p += psnr(pred, full) * bs
                s += ssim(pred, full) * bs
                r += rmse(pred, full) * bs
                n += bs

        wandb.log({
            'epoch':      epoch,
            'train/loss': train_loss,
            'val/psnr':   p / n,
            'val/ssim':   s / n,
            'val/rmse':   r / n,
        })

    run.finish()


# Launch the sweep agent
wandb.agent(sweep_id, function=train_run, count=SWEEP_COUNT)

### Viewing Results

Open [wandb.ai](https://wandb.ai) → your project `ctdenoiser` → **Sweeps** tab.

Useful views:
- **Parallel coordinates** — see which HP combos dominate on val/psnr
- **Scatter plot** `model` vs `val/psnr` — direct RED-CNN vs CTformer comparison
- **Importance** panel — which hyperparameter matters most